# Prepare SQL Server sensor data

Read `dbo.CleanedSensorData` with the project's existing `.env` database settings.
The steps below follow Notebook 02 for sensor aliases, A1/A2 consolidation,
missingness plots, 10-minute forward filling, fault cleaning, and export masks.

The export cells write the source snapshot, prepared CSV, and four matching
masks to `docs/raw/sql_server/` by default. Run the cells in order when ready.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv

# Support launching Jupyter from either the repository root or notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "database.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
from src.database import get_connection, close_connection


## Configure the SQL query and export directory

In [ ]:
START_TIME = "2026-04-14 00:00:00"  # None uses the earliest available reading.
END_TIME = "2026-04-23 23:59:00"    # Inclusive minute. None uses the latest reading.
EXPORT_DIR = PROJECT_ROOT / "docs" / "raw" / "sql_server"

start = pd.Timestamp(START_TIME) if START_TIME is not None else None
end = pd.Timestamp(END_TIME) if END_TIME is not None else None
for boundary in (start, end):
    if boundary is not None and (
        pd.isna(boundary) or boundary.tz is not None or boundary != boundary.floor("min")
    ):
        raise ValueError("Use timezone-naive, minute-aligned query boundaries.")
if start is not None and end is not None and start > end:
    raise ValueError("START_TIME must be at or before END_TIME.")

conditions = []
parameters = []
if start is not None:
    conditions.append("[datetime] >= ?")
    parameters.append(start.to_pydatetime())
if end is not None:
    conditions.append("[datetime] < ?")
    parameters.append((end + pd.Timedelta(minutes=1)).to_pydatetime())

query = "SELECT * FROM [dbo].[CleanedSensorData]"
if conditions:
    query += " WHERE " + " AND ".join(conditions)
query += " ORDER BY [datetime]"


In [ ]:
connection = get_connection()
try:
    cursor = connection.cursor()
    try:
        cursor.execute(query, *parameters)
        columns = [column[0] for column in cursor.description]
        # Convert ODBC rows to tuples for a consistent pandas tabular shape.
        df = pd.DataFrame.from_records(
            [tuple(row) for row in cursor], columns=columns
        )
    finally:
        cursor.close()
finally:
    close_connection(connection)

# SQL NULL values remain missing.
df.columns = df.columns.str.strip()
timestamp_columns = [column for column in df.columns if column.casefold() == 'datetime']
if len(timestamp_columns) != 1:
    raise ValueError('Expected exactly one datetime column in the SQL result.')
timestamp_column = timestamp_columns[0]
df[timestamp_column] = pd.to_datetime(df[timestamp_column], errors='raise')
df = df.set_index(timestamp_column).sort_index()
df.index.name = 'datetime'
if df.empty:
    raise ValueError('The SQL query returned no readings. Review the time window.')
if df.index.hasnans or df.index.tz is not None or not df.index.equals(df.index.floor('min')):
    raise ValueError('Expected nonmissing, timezone-naive, minute-aligned SQL timestamps.')
if not df.index.is_unique or not df.columns.is_unique:
    raise ValueError('Expected unique SQL timestamps and sensor columns.')

sql_raw = df.copy()
print(f"Loaded {len(df):,} rows and {len(df.columns)} sensor columns.")
display(df.head())


In [ ]:
sensor_config_path = PROJECT_ROOT / 'Sensor_Configurations.xlsx'
alias_reference = pd.read_excel(
    sensor_config_path, sheet_name='Sheet1', header=27, usecols='D:E'
).rename(columns=str.strip)
alias_reference = alias_reference.dropna(
    subset=['Prefered Alias', 'Full Tag Name']
).copy()
alias_reference['Prefered Alias'] = alias_reference['Prefered Alias'].str.strip()
alias_reference['Full Tag Name'] = alias_reference['Full Tag Name'].str.strip()
alias_reference = alias_reference.loc[
    alias_reference['Prefered Alias'].ne('')
    & alias_reference['Full Tag Name'].ne('')
]
alias_reference['tag_key'] = alias_reference['Full Tag Name'].str.casefold()
if alias_reference['tag_key'].duplicated().any():
    duplicates = alias_reference.loc[
        alias_reference['tag_key'].duplicated(keep=False), 'Full Tag Name'
    ].tolist()
    raise ValueError(f'Duplicate full tag names in Sensor_Configurations.xlsx: {duplicates}')
if alias_reference['Prefered Alias'].duplicated().any():
    duplicates = alias_reference.loc[
        alias_reference['Prefered Alias'].duplicated(keep=False), 'Prefered Alias'
    ].tolist()
    raise ValueError(f'Duplicate preferred aliases in Sensor_Configurations.xlsx: {duplicates}')

alias_by_tag = alias_reference.set_index('tag_key')['Prefered Alias'].to_dict()
# SQL imports may already use aliases or the older airflow column names.
alias_lookup = {alias.casefold(): alias for alias in alias_by_tag.values()}
alias_lookup.update(alias_by_tag)
alias_lookup.update({'q_a1_air': 'A1_Q_air', 'q_a2_air': 'A2_Q_air'})
rename_map = {
    column: alias_lookup[column.casefold()]
    for column in df.columns
    if column.casefold() in alias_lookup and column != alias_lookup[column.casefold()]
}
unmatched_columns = [column for column in df.columns if column.casefold() not in alias_lookup]
df = df.rename(columns=rename_map)
if not df.columns.is_unique:
    raise ValueError('Preferred aliases produced duplicate dataframe columns.')

print(f'Renamed {len(rename_map)} sensor columns using Sensor_Configurations.xlsx.')
if unmatched_columns:
    print('Columns without a preferred alias were left unchanged:', unmatched_columns)


## Combine paired A1/A2 tank sensors

In [ ]:
sensor_pairs = {
    'A_DO': ('A1_A_DO', 'A2_A_DO'),
    'O_DO': ('A1_O_DO', 'A2_O_DO'),
    'A_MLSS': ('A1_A_MLSS', 'A2_A_MLSS'),
    'O_MLSS': ('A1_O_MLSS', 'A2_O_MLSS'),
    'A_ORP': ('A1_A_ORP', 'A2_A_ORP'),
    'O_ORP': ('A1_O_ORP', 'A2_O_ORP'),
    'A_PH': ('A1_A_PH', 'A2_A_PH'),
    'O_PH': ('A1_O_PH', 'A2_O_PH'),
}

source_columns = [column for pair in sensor_pairs.values() for column in pair]
missing_columns = [column for column in source_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing sensor columns required for consolidation: {missing_columns}')

overlap_counts = {}
for combined_column, (a1_column, a2_column) in sensor_pairs.items():
    overlap_counts[combined_column] = int(
        (df[a1_column].notna() & df[a2_column].notna()).sum()
    )
    df[combined_column] = df[[a1_column, a2_column]].mean(axis=1, skipna=True)

df = df.drop(columns=source_columns)
print(f'Combined {len(source_columns)} A1/A2 sensor columns into {len(sensor_pairs)} tank columns.')
print('Rows where both paired sensors were populated:', overlap_counts)

In [ ]:
display(df.describe())
print(f'Dataframe shape: {df.shape}')

## Plot missing readings

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean().mul(100),
}).sort_values('missing_percent', ascending=True)

fig, ax = plt.subplots(figsize=(11, max(6, 0.38 * len(missing_summary))))
bars = ax.barh(
    missing_summary.index,
    missing_summary['missing_percent'],
    color='steelblue',
)
labels = [
    f'{percent:.1f}% ({count:,})'
    for count, percent in missing_summary[['missing_count', 'missing_percent']].itertuples(index=False)
]
ax.bar_label(bars, labels=labels, padding=4, fontsize=9)
max_percent = missing_summary['missing_percent'].max()
ax.set_xlim(0, min(100, max(5, max_percent * 1.3)))
ax.set_title('Missing data by sensor column')
ax.set_xlabel('Missing values (%)')
ax.set_ylabel('Sensor column')
ax.xaxis.grid(True, linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## Fill gaps up to 10 minutes

In [ ]:
if not isinstance(df.index, pd.DatetimeIndex):
    raise TypeError('Forward filling by minutes requires a DatetimeIndex.')
if not df.index.is_unique:
    raise ValueError('Resolve duplicate timestamps before forward filling.')

minute_index = pd.date_range(
    df.index.min(), df.index.max(), freq='min', name=df.index.name
)
df_minute = df.reindex(minute_index)
df_ffill_10m = df_minute.ffill(limit=10)
df_ffill_10m_mask = df_minute.isna() & df_ffill_10m.notna()

print(f'Original cleaned dataframe shape: {df.shape}')
print(f'Complete minute-grid shape: {df_minute.shape}')
print(f'10-minute forward-fill shape: {df_ffill_10m.shape}')

In [ ]:
missing_counts = df_ffill_10m.isna().sum().sort_values(ascending=True)
missing_percent = missing_counts.div(len(df_minute)).mul(100)
sensor_order = missing_counts.index

positions = pd.Series(range(len(sensor_order)), index=sensor_order, dtype=float)
fig, ax = plt.subplots(figsize=(12, max(7, 0.5 * len(sensor_order))))
bars_10m = ax.barh(
    positions, missing_counts, color='darkorange',
)
labels_10m = [
    f'{count:,} ({percent:.1f}%)'
    for count, percent in zip(missing_counts, missing_percent)
]
ax.bar_label(bars_10m, labels=labels_10m, padding=3, fontsize=8)
ax.set_yticks(positions, sensor_order)
max_missing = missing_counts.max()
ax.set_xlim(0, max(10, max_missing * 1.3))
ax.set_title('Remaining missing data after 10-minute forward filling')
ax.set_xlabel('Missing rows')
ax.set_ylabel('Sensor column')
ax.xaxis.grid(True, linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## Identify and clean sensor faults

In [ ]:
# Clean faults after the selected 10-minute forward fill and before plotting.
SYNCHRONIZED_FROZEN_SENSOR_FRACTION = 0.80
SYNCHRONIZED_MINUTES = 60
LONG_FLATLINE_MINUTES = 60
SHORT_ERROR_MAX_MINUTES = 10
HAMPEL_WINDOW_MINUTES = 61
HAMPEL_SIGMA = 6


def mask_runs_at_least(mask, minimum_length):
    run_id = mask.ne(mask.shift()).cumsum()
    run_length = mask.groupby(run_id).transform('size')
    return mask & run_length.ge(minimum_length)


def mask_runs_at_most(mask, maximum_length):
    run_id = mask.ne(mask.shift()).cumsum()
    run_length = mask.groupby(run_id).transform('size')
    return mask & run_length.le(maximum_length)


# Drop long intervals where most non-constant sensors freeze at the same time.
informative_columns = df_ffill_10m.columns[df_ffill_10m.nunique(dropna=True).gt(1)]
unchanged = df_ffill_10m[informative_columns].eq(
    df_ffill_10m[informative_columns].shift()
)
synchronized_candidate = unchanged.mean(axis=1).ge(
    SYNCHRONIZED_FROZEN_SENSOR_FRACTION
)
synchronized_corrupt_mask = mask_runs_at_least(
    synchronized_candidate, SYNCHRONIZED_MINUTES
)
# Include the first timestamp whose value is subsequently repeated.
synchronized_corrupt_mask |= synchronized_corrupt_mask.shift(
    -1, fill_value=False
)
df_working = df_ffill_10m.loc[~synchronized_corrupt_mask].copy()

# Treat per-sensor long flatlines as faults without discarding healthy sensors.
long_flatline_mask = pd.DataFrame(False, index=df_working.index, columns=df_working.columns)
for column in df_working.columns:
    values = df_working[column]
    run_id = values.ne(values.shift()).cumsum()
    run_length = run_id.map(run_id.value_counts())
    long_flatline_mask[column] = values.notna() & run_length.ge(
        LONG_FLATLINE_MINUTES
    )

# Review these engineering limits with the process owner before production use.
sensor_bounds = {
    'A1_COD_in': (0, None), 'A2_COD_in': (0, None),
    'A1_Q_air': (0, None), 'A2_Q_air': (0, None),
    'A1_Qin': (0, None), 'A2_Qin': (0, None),
    'A1_FQ_CH3OH': (0, None), 'A2_FQ_CH3OH': (0, None),
    'A1_Q_r1': (0, None), 'A2_Q_r1': (0, None),
    'A1_NH3_N': (0, None), 'A2_NH3_N': (0, None),
    'Eff_NH4_N': (0, None), 'Eff_NO3_N': (0, None),
    'A_DO': (0, 25), 'O_DO': (0, 25),
    'A_MLSS': (0, None), 'O_MLSS': (0, None),
    'A_PH': (0, 14), 'O_PH': (0, 14),
}
physical_fault_mask = pd.DataFrame(False, index=df_working.index, columns=df_working.columns)
for column, (lower, upper) in sensor_bounds.items():
    if column not in df_working.columns:
        continue
    if lower is not None:
        physical_fault_mask[column] |= df_working[column].lt(lower)
    if upper is not None:
        physical_fault_mask[column] |= df_working[column].gt(upper)

# Hampel candidates are retained for review; they are not automatically changed.
local_median = df_working.rolling(
    HAMPEL_WINDOW_MINUTES, center=True, min_periods=20
).median()
absolute_deviation = (df_working - local_median).abs()
local_mad = absolute_deviation.rolling(
    HAMPEL_WINDOW_MINUTES, center=True, min_periods=20
).median()
hampel_candidate_mask = (
    absolute_deviation.gt(HAMPEL_SIGMA * 1.4826 * local_mad)
    & local_mad.gt(0)
)

# Only short, physically invalid runs are considered confirmed automatic repairs.
confirmed_short_error_mask = physical_fault_mask.apply(
    mask_runs_at_most, maximum_length=SHORT_ERROR_MAX_MINUTES
)
sensor_fault_mask = long_flatline_mask | physical_fault_mask
df_masked = df_working.mask(sensor_fault_mask)
interpolated = df_masked.interpolate(
    method='time', limit=SHORT_ERROR_MAX_MINUTES, limit_area='inside'
)
short_error_replacement_mask = confirmed_short_error_mask & interpolated.notna()
df_clean = df_masked.where(~short_error_replacement_mask, interpolated)

cleaning_summary = pd.DataFrame({
    'long_flatline_cells': long_flatline_mask.sum(),
    'physical_fault_cells': physical_fault_mask.sum(),
    'hampel_candidates_for_review': hampel_candidate_mask.sum(),
    'short_errors_replaced': short_error_replacement_mask.sum(),
    'remaining_missing_cells': df_clean.isna().sum(),
})
print(f'Dropped {synchronized_corrupt_mask.sum():,} synchronized corrupt timestamps.')
display(cleaning_summary)

## Compare original and cleaned time series

In [ ]:
def plot_sensor_readings(frame, title, color):
    columns = list(frame.columns)
    ncols = 2
    nrows = (len(columns) + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(18, 2.8 * nrows), sharex=True, squeeze=False
    )
    for ax, column in zip(axes.flat, columns):
        ax.plot(frame.index, frame[column], color=color, linewidth=0.7)
        ax.set_title(column)
        ax.set_ylabel('Reading')
        ax.grid(True, linestyle='--', alpha=0.25)
    for ax in axes.flat[len(columns):]:
        ax.set_visible(False)
    for ax in axes[-1, :]:
        if ax.get_visible():
            ax.set_xlabel('Datetime')
    fig.suptitle(title, fontsize=16, y=1.0)
    fig.tight_layout()
    plt.show()


plot_sensor_readings(
    df_clean, 'Cleaned sensor readings after 10-minute forward fill', 'green'
)

In [ ]:
print(df.columns)
print(df.shape)

## Review and remove configured constant sensors

In [ ]:
constant_sensor_columns = ['A1_NH3_N', 'A2_NH3_N', 'Eff_NO3_N']
missing_constant_columns = [
    column for column in constant_sensor_columns if column not in df_clean.columns
]
if missing_constant_columns:
    raise ValueError(f'Expected constant sensors are missing: {missing_constant_columns}')

unexpected_variation = {
    column: int(df_clean[column].nunique(dropna=True))
    for column in constant_sensor_columns
    if df_clean[column].nunique(dropna=True) > 1
}
if unexpected_variation:
    raise ValueError(
        f'Review constant-sensor exclusions because variation was found: {unexpected_variation}'
    )

df_clean = df_clean.drop(columns=constant_sensor_columns)
df_ffill_10m_mask = df_ffill_10m_mask.drop(columns=constant_sensor_columns)
sensor_fault_mask = sensor_fault_mask.drop(columns=constant_sensor_columns)
short_error_replacement_mask = short_error_replacement_mask.drop(
    columns=constant_sensor_columns
)
print(f'Dropped constant sensors before export: {constant_sensor_columns}')
print(f'Export shape after constant-sensor removal: {df_clean.shape}')

## Save the SQL snapshot, prepared data, and masks

In [ ]:
data_filled_path = EXPORT_DIR / 'data_filled.csv'
data_filled_mask_path = EXPORT_DIR / 'data_filled_mask.csv'
sensor_fault_mask_path = EXPORT_DIR / 'data_sensor_fault_mask.csv'
short_error_mask_path = EXPORT_DIR / 'data_short_error_replacement_mask.csv'
synchronized_mask_path = EXPORT_DIR / 'data_synchronized_corrupt_mask.csv'
data_filled_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(
    data_filled_path, index_label='datetime', encoding='utf-8-sig'
)
df_ffill_10m_mask.loc[df_clean.index].to_csv(
    data_filled_mask_path, index_label='datetime', encoding='utf-8-sig'
)
sensor_fault_mask.to_csv(
    sensor_fault_mask_path, index_label='datetime', encoding='utf-8-sig'
)
short_error_replacement_mask.to_csv(
    short_error_mask_path, index_label='datetime', encoding='utf-8-sig'
)
synchronized_corrupt_mask.rename('synchronized_corrupt').to_csv(
    synchronized_mask_path, index_label='datetime', encoding='utf-8-sig'
)

saved_df = pd.read_csv(
    data_filled_path, index_col='datetime', parse_dates=True, encoding='utf-8-sig'
)
if saved_df.shape != df_clean.shape:
    raise ValueError(
        f'Saved shape {saved_df.shape} does not match source shape {df_clean.shape}.'
    )
if saved_df.columns.tolist() != df_clean.columns.tolist():
    raise ValueError('Saved sensor columns do not match df_clean.')
if not saved_df.index.equals(df_clean.index):
    raise ValueError('Saved datetime index does not match df_clean.')
saved_mask = pd.read_csv(
    data_filled_mask_path, index_col='datetime', parse_dates=True,
    encoding='utf-8-sig', dtype=str,
)
if not saved_mask.index.equals(saved_df.index):
    raise ValueError('Saved fill-mask index does not match data_filled.csv.')
if saved_mask.columns.tolist() != saved_df.columns.tolist():
    raise ValueError('Saved fill-mask columns do not match data_filled.csv.')
if not saved_mask.isin(['True', 'False']).all().all():
    raise ValueError('Saved fill mask contains values other than True or False.')

print(f'Saved {saved_df.shape[0]:,} rows and {saved_df.shape[1]} sensor columns to {data_filled_path}.')
print(f'Saved matching 10-minute fill mask to {data_filled_mask_path}.')
print(f'Saved sensor fault mask to {sensor_fault_mask_path}.')
print(f'Saved short-error replacement mask to {short_error_mask_path}.')
print(f'Saved synchronized corrupt-interval mask to {synchronized_mask_path}.')

sql_raw_path = EXPORT_DIR / 'sql_raw.csv'
sql_raw.to_csv(sql_raw_path, index_label='datetime', encoding='utf-8-sig')
print(f'Saved the original SQL readings to {sql_raw_path}.')
